In [1]:
# =============================================================================
#  STOCK WEEKLY TREND CLASSIFICATION  [v1 — designed for 70%+]
#
#  Setup để đảm bảo kết quả có ý nghĩa và khả thi:
#
#  1. ĐƠN VỊ: TUẦN (W-FRI resample)
#     - Noise ngày triệt tiêu nhau qua 5 ngày
#     - ~370 tuần train → đủ signal
#     - Trend tuần dễ predict hơn ngày nhiều
#
#  2. CHỈ 2 CLASS: UP vs DOWN
#     - Bỏ sideways → model không bị confuse với vùng mơ hồ
#     - Chỉ label tuần có |ret| > SIGNAL_THR (2%) → loại bỏ tuần noise
#     - Sideways tuần bị drop khỏi dataset → tập trung học signal sắc nét
#
#  3. TARGET ĐÚNG NGHĨA:
#     - ret_next_week = close[tuần t+1] / close[tuần t] - 1
#     - Dùng CLOSE (giá đóng cửa) thay vì mid — chuẩn hơn về tài chính
#
#  4. FEATURES CHẤT LƯỢNG CAO:
#     - Weekly OHLCV aggregation
#     - Multi-timeframe: 4w, 8w, 13w, 26w (1/2/3/6 tháng)
#     - Momentum, trend strength, volume confirmation
#     - Chỉ dùng data của tuần t-1 trở về trước → leakage-free
#
#  5. WALK-FORWARD EXPANDING WINDOW:
#     - Fold 0: train[0..n_init] → val fold 1
#     - Fold 1: fine-tune → val fold 2
#     - ...
#     - Mỗi fold ~10 tuần (~2.5 tháng)
#
#  4 PHASES:
#    P1: LightGBM + RF Ensemble
#    P2: LightGBM + RF + STL features
#    P3: BiLSTM + Attention
#    P4: BiLSTM + STL features
# =============================================================================

import warnings, os, random, gc
warnings.filterwarnings('ignore')
os.environ['CUDA_VISIBLE_DEVICES']      = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL']      = '3'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'false'

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    print(f'TensorFlow GPU enabled: {len(gpus)} device(s)')
else:
    print('TensorFlow GPU not found, using CPU')
from tensorflow.keras import layers, Model, Input, callbacks, optimizers, regularizers
from tensorflow.keras.utils import to_categorical

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 9,
})

# =============================================================================
# CONFIG
# =============================================================================
DATA_PATH_FPT = "/kaggle/input/datasets/cdnghnam/fptckhon/FPT raw.csv"
DATA_PATH_CMC = "/kaggle/input/datasets/cdnghnam/cmcckhon/CMC raw.csv"

# Ngưỡng signal: chỉ học tuần có |weekly_ret| > SIGNAL_THR
# Tuần nằm trong [-SIGNAL_THR, +SIGNAL_THR] bị drop → giảm noise
SIGNAL_THR = 2.0   # % — tuần tăng/giảm > 2% mới label, còn lại bỏ

CLASSES      = ['up', 'down']
CLASS_COLORS = {'up': '#1B5E20', 'down': '#B71C1C'}

# Walk-forward
N_FOLDS   = 5      # 5 folds, mỗi fold ~10 tuần
INIT_FRAC = 0.75   # 75% đầu làm init train

# STL
STL_ORDER = (1, 1, 1)
STL_WIN   = 104  # 2 năm tuần

# LSTM
SEQ_LEN      = 12   # nhìn 12 tuần trước (3 tháng)
LSTM_UNITS   = 48
LSTM_DROPOUT = 0.3
LSTM_EPOCHS  = 80
LSTM_BATCH   = 16
LSTM_LR      = 5e-4

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

print("=" * 72)
print(f"  Weekly Trend Classification  [2-class, signal_thr={SIGNAL_THR}%]")
print(f"  Logic: chỉ label tuần có |ret|>{SIGNAL_THR}% → model học signal sắc nét")
print("=" * 72)


# =============================================================================
# PHẦN 1: LOAD & RESAMPLE SANG TUẦN
# =============================================================================

def load_weekly(path, high_col, low_col, vol_col, close_col, date_col='Date'):
    """
    Load daily data → resample sang weekly (W-FRI).
    Close của tuần = close của ngày cuối tuần (thứ Sáu).
    """
    df = pd.read_csv(path, parse_dates=[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)
    df[df.columns[1:]] = df[df.columns[1:]].ffill().bfill()
    df = df.rename(columns={
        date_col: 'date', close_col: 'close',
        high_col: 'high', low_col: 'low', vol_col: 'volume'
    })
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')[['close', 'high', 'low', 'volume']]
    df['volume'] = df['volume'] / 1e6

    # Weekly OHLCV: close = last, high = max, low = min, vol = sum
    wk = df.resample('W-FRI').agg(
        close  = ('close',  'last'),
        high   = ('high',   'max'),
        low    = ('low',    'min'),
        volume = ('volume', 'sum'),
        open   = ('close',  'first'),   # open tuần = close ngày đầu tuần
    ).dropna()
    wk = wk.reset_index()
    wk['mid'] = (wk['high'] + wk['low']) / 2
    return wk


# =============================================================================
# PHẦN 2: LABEL — chỉ giữ tuần có signal rõ
# =============================================================================

def label_weekly(df_w, signal_thr=SIGNAL_THR):
    """
    weekly_ret = close[t] / close[t-1] - 1
    next_ret   = close[t+1] / close[t] - 1  ← TARGET

    Label: up nếu next_ret > +signal_thr%
           down nếu next_ret < -signal_thr%
           sideways → DROP (không label, không train/test)

    Tại sao drop sideways?
      - Tuần sideways (-2%..+2%) là noise, không có directional signal
      - Model cố học sideways → confuse toàn bộ decision boundary
      - Bỏ đi → tập trung vào tuần có trend rõ → accuracy cao hơn nhiều
    """
    df = df_w.copy()
    df['weekly_ret']  = df['close'].pct_change() * 100
    df['next_ret']    = df['weekly_ret'].shift(-1)
    df['next_close']  = df['close'].shift(-1)

    # Label dựa trên next_ret
    df['label'] = np.where(df['next_ret'] >  signal_thr, 'up',
                  np.where(df['next_ret'] < -signal_thr, 'down', 'sideways'))

    # Drop sideways và NaN
    df = df[df['label'] != 'sideways'].dropna(subset=['next_ret', 'label'])
    df = df.reset_index(drop=True)

    print(f"  After dropping sideways (|ret|<{signal_thr}%): {len(df)} weeks")
    print(f"  Class dist: {df['label'].value_counts().to_dict()}")
    up_pct = (df['label'] == 'up').mean() * 100
    print(f"  Up rate: {up_pct:.1f}%  Down rate: {100-up_pct:.1f}%")
    return df


# =============================================================================
# PHẦN 3: FEATURE ENGINEERING (weekly, leakage-free)
#
# Tất cả features tính trên close.shift(1) — không dùng thông tin tuần hiện tại.
# Điểm t chỉ nhìn [t-1, t-2, ...] để predict next_ret của t+1.
# =============================================================================

def build_weekly_features(df_w):
    """
    ~35 features tuần, leakage-free.
    Vì df_w đã drop sideways, index không liên tục → cần reset trước khi shift.
    """
    # Reset để shift chính xác theo vị trí
    f = df_w.copy().reset_index(drop=True)

    # Giá tham chiếu: close tuần trước (shift 1)
    c = f['close'].shift(1)   # leakage-free
    h = f['high'].shift(1)
    l = f['low'].shift(1)
    v = f['volume'].shift(1)
    r = f['weekly_ret'].shift(1)   # return tuần trước

    # ── Returns nhiều khung thời gian ─────────────────────────────────────────
    for w in [1, 2, 3, 4, 8, 13, 26]:
        f[f'ret_{w}w'] = c.pct_change(w) * 100

    # ── Momentum (close t-1 / MA) ──────────────────────────────────────────────
    for w in [4, 8, 13, 26, 52]:
        ma = c.rolling(w, min_periods=2).mean()
        f[f'mom_{w}w']     = (c / ma.replace(0, np.nan) - 1) * 100
        f[f'ma_std_{w}w']  = c.rolling(w, min_periods=2).std()

    # ── EMA crossover ─────────────────────────────────────────────────────────
    ema4  = c.ewm(span=4,  adjust=False).mean()
    ema13 = c.ewm(span=13, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    f['ema_cross_4_13']  = (ema4  - ema13) / (ema13.abs() + 1e-9) * 100
    f['ema_cross_13_26'] = (ema13 - ema26) / (ema26.abs() + 1e-9) * 100

    # ── MACD weekly ───────────────────────────────────────────────────────────
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26e = c.ewm(span=26, adjust=False).mean()
    macd   = ema12 - ema26e
    sig    = macd.ewm(span=9, adjust=False).mean()
    f['macd']      = macd / (c.abs() + 1e-9) * 100
    f['macd_sig']  = sig  / (c.abs() + 1e-9) * 100
    f['macd_hist'] = (macd - sig) / (c.abs() + 1e-9) * 100

    # ── RSI weekly ────────────────────────────────────────────────────────────
    for w in [6, 14]:
        delta = c.diff()
        gain  = delta.clip(lower=0).rolling(w, min_periods=1).mean()
        loss  = (-delta.clip(upper=0)).rolling(w, min_periods=1).mean()
        f[f'rsi_{w}w'] = 100 - 100 / (1 + gain / loss.replace(0, 0.001))

    # ── Bollinger Band ────────────────────────────────────────────────────────
    for w in [8, 20]:
        ma  = c.rolling(w, min_periods=2).mean()
        std = c.rolling(w, min_periods=2).std().fillna(1e-9)
        ub, lb = ma + 2*std, ma - 2*std
        f[f'bb_pos_{w}w']   = (c - lb) / (ub - lb + 1e-9)
        f[f'bb_width_{w}w'] = (ub - lb) / (ma.abs() + 1e-9) * 100

    # ── Volatility (rolling std of weekly returns) ────────────────────────────
    for w in [4, 8, 13]:
        f[f'vol_{w}w'] = r.rolling(w, min_periods=2).std()

    # ── High-Low range (weekly candle body) ───────────────────────────────────
    f['hl_range_pct'] = (h - l) / (c.abs() + 1e-9) * 100
    f['body_pct']     = (f['close'].shift(1) - f['open'].shift(1)) / \
                        (c.abs() + 1e-9) * 100   # bullish/bearish candle

    # ── Volume features ───────────────────────────────────────────────────────
    for w in [4, 8, 13]:
        vm = v.rolling(w, min_periods=1).mean()
        f[f'vol_ratio_{w}w'] = v / (vm + 1e-9)
    f['vol_chg_1w'] = v.pct_change() * 100
    f['vol_chg_4w'] = v.pct_change(4) * 100

    # ── Streak: chuỗi tuần tăng/giảm liên tiếp ───────────────────────────────
    prev_ret = r.fillna(0).values
    up_s = np.zeros(len(f)); dn_s = np.zeros(len(f))
    u = d = 0
    for i, rv in enumerate(prev_ret):
        if   rv > 0: u += 1; d = 0
        elif rv < 0: d += 1; u = 0
        up_s[i] = u; dn_s[i] = d
    f['up_streak']   = up_s
    f['down_streak'] = dn_s

    # ── Trend strength: R² của linear fit 8 tuần gần nhất ────────────────────
    for w in [8, 13]:
        def rolling_r2(s, window):
            out = np.zeros(len(s))
            sv  = s.values
            for i in range(len(s)):
                seg = sv[max(0, i-window+1):i+1]
                if len(seg) < 3:
                    out[i] = 0; continue
                x = np.arange(len(seg))
                p = np.polyfit(x, seg, 1)
                ss_res = np.sum((seg - np.polyval(p, x))**2)
                ss_tot = np.sum((seg - seg.mean())**2)
                out[i] = 1 - ss_res/(ss_tot + 1e-9)
            return out
        f[f'trend_r2_{w}w'] = rolling_r2(c, w)

    # ── Calendar ──────────────────────────────────────────────────────────────
    f['month_sin'] = np.sin(2 * np.pi * f['date'].dt.month / 12)
    f['month_cos'] = np.cos(2 * np.pi * f['date'].dt.month / 12)
    f['week_sin']  = np.sin(2 * np.pi * f['date'].dt.isocalendar().week.astype(float) / 52)
    f['week_cos']  = np.cos(2 * np.pi * f['date'].dt.isocalendar().week.astype(float) / 52)

    # ── Lag labels (up/down của tuần trước) ───────────────────────────────────
    for n in [1, 2, 3]:
        f[f'label_lag_{n}'] = (f['label'] == 'up').astype(float).shift(n)
    f['up_last_4w'] = (f['label'] == 'up').astype(float).shift(1).rolling(4, min_periods=1).sum()
    f['up_last_8w'] = (f['label'] == 'up').astype(float).shift(1).rolling(8, min_periods=1).sum()

    f = f.replace([np.inf, -np.inf], np.nan).ffill().bfill().fillna(0)

    exclude = ['date', 'close', 'high', 'low', 'low', 'volume', 'open', 'mid',
               'weekly_ret', 'next_ret', 'next_close', 'label']
    X = f.drop(columns=[c2 for c2 in exclude if c2 in f.columns], errors='ignore')
    y = f['label']
    return X, y


# =============================================================================
# PHẦN 4: STL DENOISING (record-level trên weekly close)
# =============================================================================

class STLDecomposer:
    def __init__(self, period=7, seasonal_window=7, trend_window=None, lowpass_window=None,
                 inner_iter=2, outer_iter=1, degree=1):
        self.period = int(period)
        self.seasonal_window = self._odd(max(3, int(seasonal_window)))
        if trend_window is None:
            tw = int(np.ceil(1.5 * self.period / (1 - 1.5 / self.seasonal_window)))
            self.trend_window = self._odd(max(3, tw))
        else:
            self.trend_window = self._odd(max(3, int(trend_window)))
        self.lowpass_window = self._odd(max(3, self.period if lowpass_window is None else int(lowpass_window)))
        self.inner_iter = max(1, int(inner_iter))
        self.outer_iter = max(0, int(outer_iter))
        self.degree = 1 if degree != 0 else 0

    @staticmethod
    def _odd(v):
        return v if v % 2 == 1 else v + 1

    @staticmethod
    def _tricube(u):
        au = np.abs(u)
        w = np.zeros_like(au)
        m = au < 1
        w[m] = (1 - au[m] ** 3) ** 3
        return w

    @staticmethod
    def _bisquare(u):
        au = np.abs(u)
        w = np.zeros_like(au)
        m = au < 1
        w[m] = (1 - au[m] ** 2) ** 2
        return w

    def _loess(self, y, window, robust_weights=None):
        y = np.asarray(y, dtype=float)
        n = len(y)
        x = np.arange(n, dtype=float)
        fit = np.empty(n, dtype=float)
        half = window // 2
        if robust_weights is None:
            robust_weights = np.ones(n, dtype=float)
        for i in range(n):
            l = max(0, i - half)
            r = min(n - 1, i + half)
            if r - l + 1 < window:
                l = max(0, r - window + 1)
                r = min(n - 1, l + window - 1)
            idx = np.arange(l, r + 1)
            xi = x[idx]
            yi = y[idx]
            d = np.max(np.abs(xi - x[i]))
            if d == 0:
                fit[i] = y[i]
                continue
            w = self._tricube((xi - x[i]) / d) * robust_weights[idx]
            if np.all(w <= 1e-12):
                fit[i] = y[i]
                continue
            if self.degree == 0:
                fit[i] = np.sum(w * yi) / np.sum(w)
            else:
                X = np.column_stack([np.ones_like(xi), xi - x[i]])
                WX = X * w[:, None]
                beta, *_ = np.linalg.lstsq(WX.T @ X, WX.T @ yi, rcond=None)
                fit[i] = beta[0]
        return fit

    def _moving_average(self, y, w):
        y = np.asarray(y, dtype=float)
        n = len(y)
        half = w // 2
        out = np.empty(n, dtype=float)
        for i in range(n):
            l = max(0, i - half)
            r = min(n, i + half + 1)
            out[i] = y[l:r].mean()
        return out

    def _smooth_cycle(self, detrended, robust_weights):
        n = len(detrended)
        seasonal = np.zeros(n, dtype=float)
        for phase in range(self.period):
            idx = np.arange(phase, n, self.period)
            if len(idx) > 0:
                seasonal[idx] = self._loess(detrended[idx], self.seasonal_window, robust_weights[idx])
        return seasonal

    def fit_transform(self, ts):
        y = np.asarray(ts, dtype=float).reshape(-1)
        n = len(y)
        if n < 2 * self.period:
            trend = pd.Series(y).rolling(3, min_periods=1).mean().to_numpy()
            seasonal = np.zeros_like(y)
            resid = y - trend
            return seasonal.astype(np.float32), trend.astype(np.float32), resid.astype(np.float32)

        trend = self._loess(y, self.trend_window)
        seasonal = np.zeros(n, dtype=float)
        robust = np.ones(n, dtype=float)

        for _ in range(self.outer_iter + 1):
            for _ in range(self.inner_iter):
                detr = y - trend
                sraw = self._smooth_cycle(detr, robust)
                lp = self._moving_average(sraw, self.period)
                lp = self._moving_average(lp, self.period)
                lp = self._moving_average(lp, 3)
                lp = self._loess(lp, self.lowpass_window)
                seasonal = sraw - lp
                trend = self._loess(y - seasonal, self.trend_window, robust)
            resid = y - seasonal - trend
            mad = np.median(np.abs(resid))
            robust = np.ones(n) if mad <= 1e-12 else self._bisquare(resid / (6.0 * mad))

        resid = y - seasonal - trend
        return seasonal.astype(np.float32), trend.astype(np.float32), resid.astype(np.float32)


def build_stl_features(close_arr, period=13):
    print("  → STL decomposition (weekly close) ...", flush=True)
    stl = STLDecomposer(period=period, seasonal_window=7, inner_iter=5, outer_iter=20)
    seasonal, trend, resid = stl.fit_transform(close_arr)
    df = pd.DataFrame({
        'stl_trend'      : trend,
        'stl_seasonal'   : seasonal,
        'stl_resid'      : resid,
        'stl_resid_abs'  : np.abs(resid),
        'stl_detrended'  : close_arr - trend,
        'stl_trend_grad' : np.gradient(trend),
        'stl_ret'        : np.diff(trend, prepend=trend[0]) / (np.abs(trend) + 1e-9) * 100,
    })
    return df, trend, resid
# =============================================================================
# PHẦN 5: LSTM (BiLSTM + Attention)
# =============================================================================

class AttentionLayer(layers.Layer):
    def __init__(self, units=32, **kw):
        super().__init__(**kw); self.units = units

    def build(self, input_shape):
        self.W = self.add_weight((input_shape[-1], self.units), 'glorot_uniform', name='W')
        self.b = self.add_weight((self.units,), 'zeros', name='b')
        self.u = self.add_weight((self.units, 1), 'glorot_uniform', name='u')
        super().build(input_shape)

    def call(self, x):
        uit = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        ait = tf.squeeze(tf.tensordot(uit, self.u, axes=1), -1)
        a   = tf.expand_dims(tf.nn.softmax(ait, axis=1), -1)
        return tf.reduce_sum(x * a, axis=1)

    def get_config(self):
        cfg = super().get_config(); cfg['units'] = self.units; return cfg


def build_bilstm(seq_len, n_feat, n_classes=2):
    inp = Input(shape=(seq_len, n_feat))
    x   = layers.Bidirectional(
              layers.LSTM(LSTM_UNITS, return_sequences=True,
                          dropout=LSTM_DROPOUT, recurrent_dropout=0.1,
                          kernel_regularizer=regularizers.l2(1e-3)))(inp)
    x   = layers.LayerNormalization()(x)
    x   = AttentionLayer(units=LSTM_UNITS)(x)
    x   = layers.Dense(32, activation='relu',
                        kernel_regularizer=regularizers.l2(1e-3))(x)
    x   = layers.Dropout(LSTM_DROPOUT)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    m   = Model(inp, out)
    m.compile(optimizer=optimizers.Adam(LSTM_LR),
              loss='categorical_crossentropy', metrics=['accuracy'])
    return m


def make_sequences(X_arr, seq_len=SEQ_LEN):
    N, F = X_arr.shape
    seqs = np.zeros((N, seq_len, F), dtype=np.float32)
    for i in range(N):
        start = max(0, i - seq_len + 1)
        seg   = X_arr[start:i+1]
        if len(seg) < seq_len:
            pad = np.tile(seg[0:1], (seq_len - len(seg), 1))
            seg = np.concatenate([pad, seg], axis=0)
        seqs[i] = seg
    return seqs


# =============================================================================
# PHẦN 6: WALK-FORWARD EXPANDING WINDOW
#
# Pattern:
#   Fold 0: train[0..n_init-1]  full train      → val[n_init..n_init+fold_size]
#   Fold 1: train[0..n_init+fs] fine-tune/refit  → val tiếp theo
#   ...
#   Fold k: train[0..n_init+k*fs] → val[...]
# =============================================================================

def walk_forward(X_all, y_all_enc, label_list, le, n_init, n_folds, fold_size,
                 mode='ensemble', arima_cols_idx=None):
    """
    mode: 'ensemble' → LightGBM+RF
          'lstm'     → BiLSTM
    arima_cols_idx: nếu không None thì X_all bao gồm STL features
    """
    N         = len(y_all_enc)
    n_classes = len(label_list)
    all_true  = []
    all_pred  = []
    all_proba = []

    sc        = StandardScaler()
    sc.fit(X_all[:n_init])              # scaler fit 1 lần trên init train
    X_sc_all  = sc.transform(X_all).astype(np.float32)

    lgb_model = rf_model = lstm_model = None
    init_lr   = LSTM_LR

    for fold in range(n_folds):
        tr_end    = n_init + fold * fold_size
        val_start = tr_end
        val_end   = min(tr_end + fold_size, N) if fold < n_folds - 1 else N
        if val_start >= N: break

        X_tr = X_sc_all[:tr_end]
        y_tr = y_all_enc[:tr_end]
        X_va = X_sc_all[val_start:val_end]
        y_va = y_all_enc[val_start:val_end]

        n_val_fold = val_end - val_start
        tag = "FULL TRAIN" if fold == 0 else f"fine-tune/refit"
        print(f"  Fold {fold}/{n_folds-1} [{tag}] | "
              f"train[0..{tr_end-1}]({tr_end}wk) → "
              f"val[{val_start}..{val_end-1}]({n_val_fold}wk)", flush=True)

        counts = np.bincount(y_tr, minlength=n_classes)
        cw_map = {i: len(y_tr)/(n_classes * max(c,1)) for i,c in enumerate(counts)}
        sw     = np.array([cw_map[y] for y in y_tr])

        if mode == 'ensemble':
            # LightGBM + RF: re-fit expanding data (nhanh, ~2s/fold)
            lgb_model = lgb.LGBMClassifier(
                objective='binary', n_estimators=500,
                max_depth=4, num_leaves=15,
                learning_rate=0.02, subsample=0.8, colsample_bytree=0.8,
                min_child_samples=5, reg_alpha=0.1, reg_lambda=1.0,
                random_state=SEED, n_jobs=-1, device='cpu', verbose=-1)
            lgb_model.fit(X_tr, y_tr, sample_weight=sw)

            rf_model = RandomForestClassifier(
                n_estimators=300, max_depth=6, min_samples_split=6,
                min_samples_leaf=2, max_features='sqrt',
                class_weight=cw_map, random_state=SEED, n_jobs=-1)
            rf_model.fit(X_tr, y_tr)

            proba = 0.6 * lgb_model.predict_proba(X_va) + \
                    0.4 * rf_model.predict_proba(X_va)

        elif mode == 'lstm':
            y_cat   = to_categorical(y_tr, n_classes)
            X_tr_sq = make_sequences(X_tr, SEQ_LEN)
            X_va_sq = make_sequences(
                np.vstack([X_tr[-SEQ_LEN:], X_va]), SEQ_LEN
            )[SEQ_LEN:]   # sliding window: val nhìn vào cuối train

            n_val_i = max(8, int(0.12 * len(X_tr_sq)))

            if fold == 0 or lstm_model is None:
                # Full train
                tf.keras.backend.clear_session()
                lstm_model = build_bilstm(SEQ_LEN, X_tr_sq.shape[2], n_classes)
                ep       = LSTM_EPOCHS
                patience = 10
            else:
                # Fine-tune: ít epochs hơn, LR nhỏ hơn, chỉ dùng data gần nhất
                lstm_model.optimizer.learning_rate.assign(
                    float(lstm_model.optimizer.learning_rate) * 0.3)
                X_tr_sq = X_tr_sq[-fold_size*4:]   # ~40 tuần gần nhất
                y_cat   = y_cat[-fold_size*4:]
                n_val_i = max(5, int(0.12 * len(X_tr_sq)))
                ep       = 20
                patience = 5

            lstm_model.fit(
                X_tr_sq[:-n_val_i], y_cat[:-n_val_i],
                validation_data=(X_tr_sq[-n_val_i:], y_cat[-n_val_i:]),
                epochs=ep, batch_size=LSTM_BATCH,
                class_weight=cw_map,
                callbacks=[
                    callbacks.EarlyStopping(monitor='val_loss', patience=patience,
                                            restore_best_weights=True, min_delta=1e-4),
                    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                                patience=patience//2,
                                                min_lr=1e-7, verbose=0),
                ], verbose=0)
            proba = lstm_model.predict(X_va_sq, verbose=0)

        pred = proba.argmax(axis=1)
        all_true.extend(y_va.tolist())
        all_pred.extend(pred.tolist())
        all_proba.extend(proba.tolist())

        fold_acc = accuracy_score(y_va, pred)
        print(f"    acc={fold_acc:.3f} | "
              f"dist={dict(zip(label_list, np.bincount(pred, minlength=n_classes)))}",
              flush=True)

    if mode == 'lstm' and lstm_model is not None:
        del lstm_model; gc.collect(); tf.keras.backend.clear_session()

    y_true_str = le.inverse_transform(all_true)
    y_pred_str = le.inverse_transform(all_pred)
    proba_arr  = np.array(all_proba)

    # Feature importance (ensemble only)
    imp_df = None
    if mode == 'ensemble' and lgb_model is not None:
        imp_df = pd.DataFrame({
            'feature'   : [f'f{i}' for i in range(X_all.shape[1])],
            'importance': 0.6 * lgb_model.feature_importances_ +
                          0.4 * rf_model.feature_importances_,
        }).sort_values('importance', ascending=False)

    return y_true_str, y_pred_str, proba_arr, imp_df


# =============================================================================
# PHẦN 7: EVALUATION
# =============================================================================

def evaluate(y_true, y_pred, phase, label_list):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, labels=label_list,
                           average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, labels=label_list,
                        average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, labels=label_list,
                    average='macro', zero_division=0)
    print(f"\n  [{phase}]")
    print(f"    Accuracy  = {acc:.4f}  ({'✓ ≥70%' if acc>=0.70 else '✗ <70%'})")
    print(f"    Precision = {prec:.4f}  (macro)")
    print(f"    Recall    = {rec:.4f}  (macro)")
    print(f"    F1-macro  = {f1:.4f}")
    print()
    print(classification_report(y_true, y_pred, labels=label_list, zero_division=0))
    return {'phase': phase, 'accuracy': acc, 'precision': prec,
            'recall': rec, 'f1': f1}


# =============================================================================
# PHẦN 8: VISUALIZATION
# =============================================================================

def visualize(stock_name, all_results, label_list, df_labeled,
              all_preds_str, all_true_str, a_fit, a_res, imp_df=None):

    colors = ['#9E9E9E', '#1565C0', '#E65100', '#2E7D32']
    pnames = ['P1_Ensemble', 'P2_Ensemble+STL', 'P3_LSTM', 'P4_LSTM+STL']

    # V1: Accuracy + F1 comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, mt in zip(axes, ['accuracy', 'f1']):
        vals = [m[mt] for m in all_results]
        bars = ax.bar(pnames, vals, color=colors, edgecolor='white',
                      alpha=0.88, width=0.55)
        ax.axhline(0.70, color='red', ls='--', lw=1.2, label='70% target')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.01,
                    f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
        ax.set_ylim(0, 1.1)
        ax.set_title(mt.upper(), fontweight='bold')
        ax.tick_params(axis='x', rotation=25)
        ax.legend(fontsize=8)
    plt.suptitle(f'V1 — {stock_name}: Metrics (weekly 2-class, signal>{SIGNAL_THR}%)',
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/{stock_name}_V1_metrics.png', bbox_inches='tight')
    plt.close()

    # V2: Confusion matrices
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    for ax, pname, yp in zip(axes.flatten(), pnames, all_preds_str):
        cm   = confusion_matrix(all_true_str, yp, labels=label_list)
        cm_n = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
        sns.heatmap(cm_n, annot=cm, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=label_list, yticklabels=label_list,
                    vmin=0, vmax=1, cbar=False)
        acc = accuracy_score(all_true_str, yp)
        ax.set_title(f'{pname}  Acc={acc:.3f}', fontweight='bold', fontsize=9)
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.suptitle(f'V2 — {stock_name}: Confusion Matrices', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/{stock_name}_V2_confusion.png', bbox_inches='tight')
    plt.close()

    # V3: Weekly close price + STL trend + predictions
    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=False)

    dates = pd.to_datetime(df_labeled['date'].values)
    close = df_labeled['close'].values
    axes[0].plot(dates, close, color='#1565C0', lw=1.2, label='Weekly Close')
    axes[0].plot(dates, a_fit[:len(dates)], color='#E65100', lw=1,
                 ls='--', label='STL Trend')
    axes[0].set_title('Weekly Close + STL Trend', fontweight='bold')
    axes[0].set_ylabel('Price'); axes[0].legend(fontsize=8)

    axes[1].bar(dates, a_res[:len(dates)], color='#6A1B9A', alpha=0.5,
                label='STL Residual (Noise)')
    axes[1].axhline(0, color='gray', ls='--', lw=0.7)
    axes[1].set_title('STL Residuals', fontweight='bold')
    axes[1].set_ylabel('Residual'); axes[1].legend(fontsize=8)

    # Actual labels
    colors_map = {'up': '#1B5E20', 'down': '#B71C1C'}
    for i, (d, lab) in enumerate(zip(dates, df_labeled['label'].values)):
        axes[2].bar(d, 1 if lab=='up' else -1, width=5,
                    color=colors_map.get(lab, 'gray'), alpha=0.7)
    axes[2].axhline(0, color='black', lw=0.5)
    axes[2].set_title('Actual Weekly Labels (up=+1, down=-1)', fontweight='bold')
    axes[2].set_yticks([-1, 1]); axes[2].set_yticklabels(['down', 'up'])

    plt.suptitle(f'V3 — {stock_name}: Data Overview', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/{stock_name}_V3_overview.png', bbox_inches='tight')
    plt.close()

    # V4: Per-class precision/recall
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(pnames)); w = 0.35
    for ax, cls in zip(axes, label_list):
        precs = [precision_score(all_true_str, yp, labels=[cls],
                                 average='macro', zero_division=0)
                 for yp in all_preds_str]
        recs  = [recall_score(all_true_str, yp, labels=[cls],
                              average='macro', zero_division=0)
                 for yp in all_preds_str]
        ax.bar(x - w/2, precs, w, color=colors, alpha=0.85, label='Precision')
        ax.bar(x + w/2, recs,  w, color=colors, alpha=0.55, label='Recall',
               edgecolor='white')
        ax.axhline(0.70, color='red', ls='--', lw=1, alpha=0.7)
        ax.set_xticks(x); ax.set_xticklabels(pnames, rotation=20, fontsize=8)
        ax.set_ylim(0, 1.1); ax.set_title(f'Class: {cls.upper()}', fontweight='bold')
        ax.legend(fontsize=8)
    plt.suptitle(f'V4 — {stock_name}: Per-class Precision/Recall', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/{stock_name}_V4_perclass.png', bbox_inches='tight')
    plt.close()

    # V5: Feature importance (P1/P2 ensemble)
    if imp_df is not None:
        try:
            imp = imp_df.head(20)
            fig, ax = plt.subplots(figsize=(10, 7))
            ax.barh(range(len(imp)), imp['importance'].values[::-1],
                    color='#1565C0', alpha=0.8)
            ax.set_yticks(range(len(imp)))
            ax.set_yticklabels(imp['feature'].values[::-1], fontsize=8)
            ax.set_title(f'V5 — {stock_name}: Feature Importance (P1)',
                         fontweight='bold')
            plt.tight_layout()
            plt.savefig(f'{OUT_DIR}/{stock_name}_V5_importance.png',
                        bbox_inches='tight')
            plt.close()
        except Exception as e:
            print(f"  [Warning] V5 skipped: {e}")

    print(f"  → Plots saved: {OUT_DIR}/{stock_name}_V*.png")


# =============================================================================
# PHẦN 9: PIPELINE CHÍNH
# =============================================================================

def run_pipeline(df_raw_daily, stock_name='FPT'):
    print(f"\n{'='*72}")
    print(f"  {stock_name} — Weekly 2-Class Pipeline")
    print(f"{'='*72}")

    # ── 1. Resample sang tuần ─────────────────────────────────────────────────
    df_w = load_weekly(
        None,  # path không dùng (truyền df_raw_daily trực tiếp)
        None, None, None, None)   # dummy — xử lý riêng bên dưới
    # (Vì load_weekly cần path, ta làm thủ công)
    raise NotImplementedError("Dùng run_pipeline_from_df thay thế")


def run_pipeline_from_df(df_daily, stock_name='FPT'):
    """
    df_daily: DataFrame với cột date, close, high, low, volume
    """
    print(f"\n{'='*72}")
    print(f"  {stock_name} — Weekly 2-Class Walk-Forward Pipeline")
    print(f"{'='*72}")

    # ── 1. Resample sang tuần ─────────────────────────────────────────────────
    df_d = df_daily.copy()
    df_d['date'] = pd.to_datetime(df_d['date'])
    df_d = df_d.set_index('date').sort_index()
    df_d['volume'] = df_d['volume'] / 1e6

    wk = df_d.resample('W-FRI').agg(
        close  = ('close',  'last'),
        high   = ('high',   'max'),
        low    = ('low',    'min'),
        volume = ('volume', 'sum'),
        open   = ('close',  'first'),
    ).dropna().reset_index()
    wk['mid'] = (wk['high'] + wk['low']) / 2

    print(f"\n[1] Weekly data: {len(wk)} weeks | "
          f"{wk['date'].min().date()} → {wk['date'].max().date()}")

    # ── 2. Label (drop sideways) ──────────────────────────────────────────────
    print("\n[2] Labeling ...")
    df_labeled = label_weekly(wk, SIGNAL_THR)
    N          = len(df_labeled)

    if N < 50:
        print("  ERROR: Quá ít data sau khi drop sideways!")
        return None

    # ── 3. Features ───────────────────────────────────────────────────────────
    print("\n[3] Building features ...")
    X_all, y_all = build_weekly_features(df_labeled)
    feat_cols    = X_all.columns.tolist()
    print(f"  Features: {len(feat_cols)}")

    label_list = CLASSES
    le         = LabelEncoder()
    le.classes_ = np.array(label_list)
    y_enc      = le.transform(y_all.values)
    n_classes  = len(label_list)

    # ── 4. STL denoising ────────────────────────────────────────────────────
    print("\n[4] STL denoising (weekly walk-forward) ...")
    close_all           = df_labeled['close'].values
    df_arima, a_fit, a_res = build_stl_features(close_all)

    X_base  = X_all.values.astype(np.float32)
    X_arima = np.hstack([X_base, df_arima.values.astype(np.float32)])

    # ── 5. Walk-forward config ────────────────────────────────────────────────
    n_init    = int(N * INIT_FRAC)
    n_remain  = N - n_init
    fold_size = max(8, n_remain // N_FOLDS)
    actual_folds = max(1, n_remain // fold_size)

    print(f"\n[5] Walk-forward setup:")
    print(f"  Init train: {n_init} weeks | Remain: {n_remain} weeks")
    print(f"  Folds: {actual_folds} × {fold_size} weeks/fold")
    print(f"  LightGBM+RF: re-fit each fold (fast)")
    print(f"  LSTM: full train fold-0, fine-tune fold-1+")

    all_results  = []
    all_preds    = []
    y_true_ref   = None
    imp_df_final = None

    # ── P1: LGB+RF ────────────────────────────────────────────────────────────
    print("\n── P1: LGB+RF Ensemble ───────────────────────────────────────────")
    y_t, y_p1, proba_p1, imp_df_final = walk_forward(
        X_base, y_enc, label_list, le,
        n_init, actual_folds, fold_size, mode='ensemble')
    y_true_ref = y_t
    m1 = evaluate(y_t, y_p1, 'P1_Ensemble', label_list)
    all_results.append(m1); all_preds.append(list(y_p1))

    # Gán tên feature cho imp_df
    if imp_df_final is not None:
        imp_df_final['feature'] = feat_cols[:len(imp_df_final)]

    # ── P2: LGB+RF + STL ────────────────────────────────────────────────────
    print("\n── P2: LGB+RF + STL features ───────────────────────────────────")
    y_t2, y_p2, proba_p2, _ = walk_forward(
        X_arima, y_enc, label_list, le,
        n_init, actual_folds, fold_size, mode='ensemble')
    m2 = evaluate(y_t2, y_p2, 'P2_Ensemble+STL', label_list)
    all_results.append(m2); all_preds.append(list(y_p2))

    # ── P3: BiLSTM ────────────────────────────────────────────────────────────
    print("\n── P3: BiLSTM + Attention ────────────────────────────────────────")
    y_t3, y_p3, proba_p3, _ = walk_forward(
        X_base, y_enc, label_list, le,
        n_init, actual_folds, fold_size, mode='lstm')
    m3 = evaluate(y_t3, y_p3, 'P3_LSTM', label_list)
    all_results.append(m3); all_preds.append(list(y_p3))

    # ── P4: BiLSTM + STL ────────────────────────────────────────────────────
    print("\n── P4: BiLSTM + STL features ───────────────────────────────────")
    y_t4, y_p4, proba_p4, _ = walk_forward(
        X_arima, y_enc, label_list, le,
        n_init, actual_folds, fold_size, mode='lstm')
    m4 = evaluate(y_t4, y_p4, 'P4_LSTM+STL', label_list)
    all_results.append(m4); all_preds.append(list(y_p4))

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*72}")
    print(f"  SUMMARY — {stock_name}  [{actual_folds} folds × {fold_size}wk]")
    print(f"  Signal threshold: |ret| > {SIGNAL_THR}%  |  2-class: up/down")
    print(f"{'='*72}")
    df_sum = pd.DataFrame(all_results)
    print(df_sum[['phase','accuracy','precision','recall','f1']].to_string(index=False))
    df_sum.to_csv(f'{OUT_DIR}/{stock_name}_weekly_summary.csv', index=False)

    best = df_sum.loc[df_sum['accuracy'].idxmax()]
    print(f"\n  Best model: {best['phase']}  acc={best['accuracy']:.4f}")
    print(f"  Target 70%: {'✓ ACHIEVED' if best['accuracy']>=0.70 else '✗ not yet'}")
    print(f"\n  ΔAcc P2−P1 = {m2['accuracy']-m1['accuracy']:+.4f}  [STL help]")
    print(f"  ΔAcc P3−P1 = {m3['accuracy']-m1['accuracy']:+.4f}  [LSTM vs Ensemble]")
    print(f"  ΔAcc P4−P3 = {m4['accuracy']-m3['accuracy']:+.4f}  [STL help for LSTM]")

    visualize(stock_name, all_results, label_list, df_labeled,
              all_preds, list(y_true_ref),
              a_fit, a_res, imp_df=imp_df_final)

    return {
        'results'    : all_results,
        'y_true'     : list(y_true_ref),
        'preds'      : {'P1': list(y_p1), 'P2': list(y_p2),
                        'P3': list(y_p3), 'P4': list(y_p4)},
        'df_labeled' : df_labeled,
        'importance' : imp_df_final,
    }


# =============================================================================
# PHẦN 10: MAIN
# =============================================================================

def load_daily(path, high_col, low_col, vol_col, close_col, date_col='Date'):
    df = pd.read_csv(path, parse_dates=[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)
    df[df.columns[1:]] = df[df.columns[1:]].ffill().bfill()
    return pd.DataFrame({
        'date'  : df[date_col],
        'close' : df[close_col].astype(float),
        'high'  : df[high_col].astype(float),
        'low'   : df[low_col].astype(float),
        'volume': df[vol_col].astype(float),
    })


print("\nLoading data ...")
df_fpt_daily = load_daily(DATA_PATH_FPT, 'High_FPT', 'Low_FPT',
                           'Volume_FPT', 'Close_FPT')
print(f"FPT daily: {len(df_fpt_daily)} rows  "
      f"({df_fpt_daily['date'].min().date()} → {df_fpt_daily['date'].max().date()})")

fpt_results = run_pipeline_from_df(df_fpt_daily, stock_name='FPT')

# ── CMC ───────────────────────────────────────────────────────────────────────
# df_cmc_daily = load_daily(DATA_PATH_CMC, 'High_CMC', 'Low_CMC',
#                            'Volume_CMC', 'Close_CMC')
# cmc_results = run_pipeline_from_df(df_cmc_daily, stock_name='CMC')

print(f"\n{'='*72}")
print("  FINAL SUMMARY")
print(f"{'='*72}")
if fpt_results:
    df_s = pd.DataFrame(fpt_results['results'])
    print(df_s[['phase','accuracy','precision','recall','f1']].to_string(index=False))
    best_acc = df_s['accuracy'].max()
    print(f"\n  Best accuracy: {best_acc:.4f}  "
          f"({'✓ ≥70%' if best_acc >= 0.70 else '✗ <70%'})")

print(f"\nAll outputs → {OUT_DIR}/")
print("=" * 72)



E0000 00:00:1779952768.122408      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779952768.183670      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779952768.644175      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779952768.644218      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779952768.644221      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779952768.644223      23 computation_placer.cc:177] computation placer already registered. Please check linka

TensorFlow GPU not found, using CPU
  Weekly Trend Classification  [2-class, signal_thr=2.0%]
  Logic: chỉ label tuần có |ret|>2.0% → model học signal sắc nét

Loading data ...
FPT daily: 1905 rows  (2017-08-24 → 2025-04-09)

  FPT — Weekly 2-Class Walk-Forward Pipeline

[1] Weekly data: 396 weeks | 2017-08-25 → 2025-04-11

[2] Labeling ...
  After dropping sideways (|ret|<2.0%): 200 weeks
  Class dist: {'up': 119, 'down': 81}
  Up rate: 59.5%  Down rate: 40.5%

[3] Building features ...
  Features: 51

[4] STL denoising (weekly walk-forward) ...
  → STL decomposition (weekly close) ...

[5] Walk-forward setup:
  Init train: 150 weeks | Remain: 50 weeks
  Folds: 5 × 10 weeks/fold
  LightGBM+RF: re-fit each fold (fast)
  LSTM: full train fold-0, fine-tune fold-1+

── P1: LGB+RF Ensemble ───────────────────────────────────────────
  Fold 0/4 [FULL TRAIN] | train[0..149](150wk) → val[150..159](10wk)
    acc=0.600 | dist={'up': np.int64(6), 'down': np.int64(4)}
  Fold 1/4 [fine-tune/refit]